<a href="https://colab.research.google.com/github/Roger-Quinelato/MDA/blob/colab/An%C3%A1lise_Comparativa_de_Palavras_por_Tipo_de_Cita%C3%A7%C3%A3o.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import nltk
from nltk.corpus import stopwords
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter
import re

# --- Análise Comparativa: Palavras-Chave para Citações Primary vs. Secondary ---
print("Iniciando a análise comparativa de palavras com uma lista de exclusão estratégica...")

# Garante que a lista de stopwords do NLTK está disponível
try:
    stopwords.words('english')
except LookupError:
    print("Baixando 'stopwords' do NLTK...")
    nltk.download('stopwords')

# Define a lista de palavras a serem ignoradas (LISTA REVISADA PARA PRESERVAR CONTEXTO)
stop_words = set(stopwords.words('english'))
palavras_extras_para_remover = {
    # Termos de formatação e citação que são quase sempre ruído
    'figure', 'fig', 'table', 'et', 'al', 'doi',
    'https', 'http', 'www', 'org',
    'also', 'however', 'supplementary', 'supplemental'
}
stop_words.update(palavras_extras_para_remover)

def get_most_common_words(df, n=15):
    """Função auxiliar para processar, limpar e contar as palavras de um DataFrame."""
    # Junta todo o texto da coluna, garantindo que não há valores nulos
    corpus = ' '.join(df['texto_do_artigo'].dropna())
    # Converte para minúsculas
    corpus = corpus.lower()
    # Encontra apenas palavras com 2 ou mais letras
    tokens = re.findall(r'\b[a-z]{2,}\b', corpus)
    # Filtra a lista de tokens, removendo as stopwords
    palavras_filtradas = [palavra for palavra in tokens if palavra not in stop_words]
    return Counter(palavras_filtradas).most_common(n)

# 1. Separa o DataFrame por tipo de citação
df_primary = df_limpo[df_limpo['type'] == 'Primary']
df_secondary = df_limpo[df_limpo['type'] == 'Secondary']

# 2. Obter as palavras mais comuns para cada tipo
primary_words = pd.DataFrame(get_most_common_words(df_primary), columns=['Palavra', 'Frequência'])
secondary_words = pd.DataFrame(get_most_common_words(df_secondary), columns=['Palavra', 'Frequência'])

# 3. Criar os gráficos de barras lado a lado para comparação
print("Gerando gráficos comparativos...")
fig, axes = plt.subplots(1, 2, figsize=(20, 8)) # 1 linha, 2 colunas de gráficos

# Gráfico para citações 'Primary'
sns.barplot(ax=axes[0], x='Frequência', y='Palavra', data=primary_words, palette='Greens_r')
axes[0].set_title('Top 15 Palavras em Citações "Primary"', fontsize=16)
axes[0].set_xlabel('Frequência')
axes[0].set_ylabel('Palavra')

# Gráfico para citações 'Secondary'
sns.barplot(ax=axes[1], x='Frequência', y='Palavra', data=secondary_words, palette='Oranges_r')
axes[1].set_title('Top 15 Palavras em Citações "Secondary"', fontsize=16)
axes[1].set_xlabel('Frequência')
axes[1].set_ylabel('') # Remove o rótulo do eixo y para um visual mais limpo

plt.tight_layout() # Ajusta o layout para evitar sobreposição
plt.show()